# 01 Data Audit — M1 완료 기준 검증

데이터 레이어(`src/data_loader.py`)의 7가지 완료 기준을 순서대로 확인한다.

In [1]:
import sys, warnings
from pathlib import Path
import pandas as pd
import yaml

# 프로젝트 루트를 sys.path에 추가
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root))

from src.data_loader import (
    load_all, validate, apply_weekly_lag,
    PRICES_CACHE, FRED_CACHE,
    _YFINANCE_TICKERS, _FRED_IDS,
)

with open(root / 'config' / 'base.yaml') as f:
    cfg = yaml.safe_load(f)

print('config:', cfg)

config: {'period': {'start': '1990-01-01', 'end': None}, 'benchmark': 'SP500TR', 'sigma_target': 0.12, 'w_max': 1.0, 'rebalance_band': 0.05, 'cost_bps': 2.0, 'borrow_spread_bps': 50, 'leverage_max': 2.0, 'exec_lag': 1, 'seed': 42, 'nfci_lag_bdays': 3, 'stlfsi_lag_bdays': 5}


## 완료 기준 1 — 데이터 로드

In [2]:
# force_refresh=False: 캐시 우선 사용 (캐시 없으면 다운로드)
warnings.simplefilter('always')
series = load_all(cfg, force_refresh=False)
print('로드된 시리즈:', list(series.keys()))

[캐시 사용] /workspaces/Strategy_Triangulate/data/raw_prices.parquet
[캐시 사용] /workspaces/Strategy_Triangulate/data/raw_fred.parquet
로드된 시리즈: ['sp500tr', 'vix', 'vix3m', 'rf', 'hy_oas', 'nfci', 'stlfsi']


## 완료 기준 2 — 시리즈별 시작·종료일·결측·중복·이상치 리포트

In [3]:
report = validate(series)
rows = []
for key, info in report.items():
    rows.append({
        '시리즈':   key,
        '시작':     info['start'],
        '종료':     info['end'],
        '유효관측': info['n_obs'],
        '결측':     info['n_missing'],
        '중복날짜': info['n_duplicate_dates'],
        '이상치':   ' | '.join(info['anomalies']),
    })
df_report = pd.DataFrame(rows)
display(df_report)
print('[VIX3M] 2006~: 짧은 표본은 사실 (기간구조 보조 피처, 오류 아님)')
print('[hy_oas] 2023-05-30~: FRED BAMLH0A0HYM2 — ICE 라이선싱 이슈로 이전 이력 삭제됨')

,시리즈,시작,종료,유효관측,결측,중복날짜,이상치
0,sp500tr,1988-01-04,2026-05-29,9674,0,0,없음
1,vix,1990-01-02,2026-05-29,9169,505,0,없음
2,vix3m,2006-07-17,2026-05-29,4999,4675,0,없음
3,rf,1988-01-04,2026-05-29,9674,0,0,없음
4,hy_oas,2023-05-30,2026-05-29,753,8921,0,⚠ 기대 시작(1996~)보다 늦음 — ICE 라이선싱 이슈
5,nfci,1988-01-06,2026-05-29,9672,2,0,없음
6,stlfsi,1994-01-07,2026-05-29,8152,1522,0,없음


[VIX3M] 2006~: 짧은 표본은 사실 (기간구조 보조 피처, 오류 아님)
[hy_oas] 2023-05-30~: FRED BAMLH0A0HYM2 — ICE 라이선싱 이슈로 이전 이력 삭제됨


## 완료 기준 3 — VXO 미혼입 · ^SP500TR 총수익 · ^GSPC 미사용 · TEDRATE 미사용

In [4]:
ticker_vals = list(_YFINANCE_TICKERS.values())
fred_ids    = list(_FRED_IDS.values())

assert '^VXO'   not in ticker_vals, 'VXO 혼입!'
assert '^GSPC'  not in ticker_vals, '^GSPC(가격지수) 혼입!'
assert 'TEDRATE' not in fred_ids,   'TEDRATE(단종) 혼입!'

print('yfinance 티커:', ticker_vals)
print('FRED ID:      ', fred_ids)

prices_raw = pd.read_parquet(PRICES_CACHE)
print('\nraw_prices 컬럼:', prices_raw.columns.tolist())
print('sp500tr (총수익 지수값):', prices_raw['sp500tr'].dropna().tail(3).to_dict())
print('\nVXO 미혼입 ✓ | ^GSPC 미사용 ✓ | ^SP500TR(총수익) 사용 ✓ | TEDRATE 미사용 ✓')

yfinance 티커: ['^SP500TR', '^VIX', '^VIX3M']
FRED ID:       ['DTB3', 'BAMLH0A0HYM2', 'NFCI', 'STLFSI4']

raw_prices 컬럼: ['sp500tr', 'vix', 'vix3m']
sp500tr (총수익 지수값): {Timestamp('2026-05-27 00:00:00'): 16800.859375, Timestamp('2026-05-28 00:00:00'): 16897.640625, Timestamp('2026-05-29 00:00:00'): 16935.349609375}

VXO 미혼입 ✓ | ^GSPC 미사용 ✓ | ^SP500TR(총수익) 사용 ✓ | TEDRATE 미사용 ✓


## 완료 기준 4 — 시장 시세 ffill 미적용

In [5]:
vix3m = series['vix3m']
vix   = series['vix']

before_vix3m = vix3m[vix3m.index < '2006-07-17']
before_vix   = vix[vix.index   < '1990-01-01']

assert before_vix3m.notna().sum() == 0, 'VIX3M ffill 감지!'
assert before_vix.notna().sum()   == 0, 'VIX ffill 감지!'

print(f'VIX3M 시작 전 비NaN: {before_vix3m.notna().sum()}건 (0이어야 함) ✓')
print(f'VIX   시작 전 비NaN: {before_vix.notna().sum()}건 (0이어야 함) ✓')
print(f'rf 결측(공표 시리즈): {series["rf"].isna().sum()}건 (0이어야 함) ✓')
print('시장 시세 ffill 미적용 확인 ✓')

VIX3M 시작 전 비NaN: 0건 (0이어야 함) ✓
VIX   시작 전 비NaN: 0건 (0이어야 함) ✓
rf 결측(공표 시리즈): 0건 (0이어야 함) ✓
시장 시세 ffill 미적용 확인 ✓


## 완료 기준 5 — 주간 시차 처리 (NFCI 금→수 +3 영업일)

In [6]:
fred_raw = pd.read_parquet(FRED_CACHE)
nfci_raw = fred_raw['nfci'].dropna()
lag      = cfg['nfci_lag_bdays']

print(f'NFCI lag: {lag} 영업일 (금→수, 시카고 연준 릴리스 스케줄)')
print('\n기준일(금요일) → 근사 공표일(수요일):')
for orig, val in nfci_raw.head(5).items():
    shifted = orig + pd.tseries.offsets.BusinessDay(lag)
    print(f'  {orig.date()} ({orig.day_name()[:3]}) → {shifted.date()} ({shifted.day_name()[:3]})  NFCI={val:.2f}')

first_pub = nfci_raw.index[0] + pd.tseries.offsets.BusinessDay(lag)
before    = series['nfci'][series['nfci'].index < first_pub]
assert before.isna().all(), '공표 전 값 누수!'
print(f'\n첫 공표일({first_pub.date()}) 이전 NaN 전수 확인 ✓ (룩어헤드 차단)')

NFCI lag: 3 영업일 (금→수, 시카고 연준 릴리스 스케줄)

기준일(금요일) → 근사 공표일(수요일):
  1971-01-08 (Fri) → 1971-01-13 (Wed)  NFCI=0.60
  1971-01-15 (Fri) → 1971-01-20 (Wed)  NFCI=0.63
  1971-01-22 (Fri) → 1971-01-27 (Wed)  NFCI=0.67
  1971-01-29 (Fri) → 1971-02-03 (Wed)  NFCI=0.70
  1971-02-05 (Fri) → 1971-02-10 (Wed)  NFCI=0.75

첫 공표일(1971-01-13) 이전 NaN 전수 확인 ✓ (룩어헤드 차단)


## 완료 기준 6 — 비거래일 공표 → 다음 거래일 push (over-shift)

In [7]:
trading_idx = series['sp500tr'].dropna().index
shifted_idx = nfci_raw.index + pd.tseries.offsets.BusinessDay(lag)
in_range    = (shifted_idx >= trading_idx[0]) & (shifted_idx <= trading_idx[-1])
non_td      = shifted_idx[in_range & ~shifted_idx.isin(trading_idx)]

print(f'거래일 범위 내 시프트 날짜: {in_range.sum()}건')
print(f'그 중 비거래일(공휴일 등): {len(non_td)}건 ← BDay over-shift 케이스')
for nd in list(non_td)[:5]:
    next_td = trading_idx[trading_idx > nd][0]
    val     = series['nfci'].iloc[trading_idx.get_loc(next_td)]
    print(f'  {nd.date()} ({nd.day_name()[:3]}, 비거래일) → {next_td.date()}: NFCI={val:.3f}')
all_ok = all(
    not pd.isna(series['nfci'].iloc[trading_idx.get_loc(trading_idx[trading_idx > nd][0])])
    for nd in non_td if len(trading_idx[trading_idx > nd]) > 0
)
assert all_ok
print(f'전 {len(non_td)}건 다음 거래일에 값 존재 ✓ (over-shift 보수적 동작)')

거래일 범위 내 시프트 날짜: 2004건
그 중 비거래일(공휴일 등): 21건 ← BDay over-shift 케이스
  1990-07-04 (Wed, 비거래일) → 1990-07-05: NFCI=-0.146
  1991-12-25 (Wed, 비거래일) → 1991-12-26: NFCI=-0.650
  1992-01-01 (Wed, 비거래일) → 1992-01-02: NFCI=-0.650
  1994-04-27 (Wed, 비거래일) → 1994-04-28: NFCI=-0.734
  1996-12-25 (Wed, 비거래일) → 1996-12-26: NFCI=-0.739
전 21건 다음 거래일에 값 존재 ✓ (over-shift 보수적 동작)


## 완료 기준 7a — 캐시 재로드 재현

In [8]:
series_cached = load_all(cfg, force_refresh=False)
for key in series:
    ok = series[key].dropna().equals(series_cached[key].dropna())
    print(f'  {key:8s}: 캐시 재현 {"✓" if ok else "✗"}')
    assert ok
print('캐시 재로드 완전 재현 ✓')

[캐시 사용] /workspaces/Strategy_Triangulate/data/raw_prices.parquet
[캐시 사용] /workspaces/Strategy_Triangulate/data/raw_fred.parquet
  sp500tr : 캐시 재현 ✓
  vix     : 캐시 재현 ✓
  vix3m   : 캐시 재현 ✓
  rf      : 캐시 재현 ✓
  hy_oas  : 캐시 재현 ✓
  nfci    : 캐시 재현 ✓
  stlfsi  : 캐시 재현 ✓
캐시 재로드 완전 재현 ✓


## 완료 기준 7b — inner join 없음 (각 시리즈 최대 기간 보존)

In [9]:
for key, s in series.items():
    v = s.dropna()
    print(f'  {key:8s}: {v.index.min().date()} ~ {v.index.max().date()}, n={len(v)}')

assert series['vix'].dropna().index.min().year    <= 1990
assert series['sp500tr'].dropna().index.min().year <= 1988
assert series['vix3m'].dropna().index.min().year  >= 2006
print('\nvix3m(2006~)이어도 vix(1990~)·sp500tr(1988~) 온전히 보존 ✓')
print('inner join 없음 ✓')

  sp500tr : 1988-01-04 ~ 2026-05-29, n=9674
  vix     : 1990-01-02 ~ 2026-05-29, n=9169
  vix3m   : 2006-07-17 ~ 2026-05-29, n=4999
  rf      : 1988-01-04 ~ 2026-05-29, n=9674
  hy_oas  : 2023-05-30 ~ 2026-05-29, n=753
  nfci    : 1988-01-06 ~ 2026-05-29, n=9672
  stlfsi  : 1994-01-07 ~ 2026-05-29, n=8152

vix3m(2006~)이어도 vix(1990~)·sp500tr(1988~) 온전히 보존 ✓
inner join 없음 ✓
